# Capstone — Refresh / Content Opportunity Scoring

This notebook ranks anonymized pages for human refresh review. It uses a transparent baseline and a leakage-safe logistic model. Results are directional decision support, not causal evidence about Google or refresh impact.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RAW_URL = 'https://raw.githubusercontent.com/Di-pesh/flyinterm/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(RAW_URL).drop_duplicates('content_id').copy()
lane = df.loc[df['content_type'].eq('keyword article')].copy()
lane['target'] = lane['trend_direction'].astype('string').str.lower().eq('down').astype(int)
print(f'Raw rows: {len(df):,}')
print(f'Keyword-article rows: {len(lane):,}')
print(f'Lane target base rate: {lane["target"].mean():.3f}')


## 1. Question and data

Which anonymized content pages should an editorial team review first for refresh or monitoring? The unit is a page and the output is a ranked queue with a score, action, and reason.

The source is the public starter snapshot `data/raw/content_refresh_anonymized.csv`. IDs are used only for grouping and tie-breaking. Label-derived fields and overlapping outcome-window fields are not model inputs.

In [ ]:
excluded = {
    'content_id', 'client_id', 'trend_direction', 'trend_pct', 'target',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'impression_tier', 'position_tier'
}
assert lane['target'].equals(lane['trend_direction'].astype('string').str.lower().eq('down').astype(int))
print(f'Excluded-field count: {len(excluded)}')
display(lane.groupby('content_type', dropna=False)['word_count'].apply(lambda s: s.isna().mean()).rename('word_count_missing_rate').to_frame())


## 2. Transparent baseline

The baseline rule is: review pages that are both stale and visible. Stale means at least 91 days since update; visible means at least 500 impressions in the reported window. The score is intentionally readable and receives a reason code.

In [ ]:
lane['is_stale'] = lane['days_since_last_update'].fillna(0).ge(91)
lane['is_visible'] = lane['impressions_90d'].fillna(0).ge(500)
lane['baseline_score'] = np.log1p(lane['impressions_90d'].fillna(0)) * (1 + lane['is_stale'].astype(int))
lane['baseline_action'] = np.where(lane['is_stale'] & lane['is_visible'], 'refresh_review', 'monitor')
lane['baseline_reason'] = np.where(lane['is_stale'] & lane['is_visible'], 'stale_visible', 'not_both_stale_visible')
baseline_queue = lane.sort_values(['baseline_score', 'content_id'], ascending=[False, True]).reset_index(drop=True)
baseline_queue.insert(0, 'rank', np.arange(1, len(baseline_queue) + 1))
display(lane['baseline_action'].value_counts().rename_axis('action').to_frame('rows'))
display(baseline_queue[['rank', 'baseline_score', 'baseline_action', 'baseline_reason', 'days_since_last_update', 'impressions_90d']].head(10))


## 3. Grouped model evaluation

The model uses metadata and recency fields only. Missing numeric fields get a training-fold median and a missingness flag; missing categories become `unknown`. A complete client is held out so pages from one client cannot occur in both train and test. Both methods are evaluated on the same test rows.

In [ ]:
safe_numeric = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']
safe_categorical = ['content_type', 'main_intent', 'age_tier', 'freshness_tier']
X_raw = lane[safe_numeric + safe_categorical].copy()
for column in safe_numeric:
    X_raw[column] = pd.to_numeric(X_raw[column], errors='coerce')
    X_raw[f'has_{column}'] = X_raw[column].notna().astype(int)
for column in safe_categorical:
    X_raw[column] = X_raw[column].astype('string')
y = lane['target'].reset_index(drop=True)
X_raw = X_raw.reset_index(drop=True)
groups = lane['client_id'].astype('string').fillna('unknown').reset_index(drop=True)
numeric_columns = safe_numeric + [f'has_{c}' for c in safe_numeric]

preprocess = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_columns),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), safe_categorical),
])
model = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))])

# Try deterministic grouped splits until both test classes are represented.
splitter = GroupShuffleSplit(n_splits=20, test_size=0.20, random_state=42)
train_idx = test_idx = None
for candidate_train, candidate_test in splitter.split(X_raw, y, groups):
    if y.iloc[candidate_train].nunique() == 2 and y.iloc[candidate_test].nunique() == 2:
        train_idx, test_idx = candidate_train, candidate_test
        break
if train_idx is None:
    raise RuntimeError('Could not find a grouped split containing both target classes.')

model.fit(X_raw.iloc[train_idx], y.iloc[train_idx])
model_scores = model.predict_proba(X_raw.iloc[test_idx])[:, 1]
baseline_scores = lane['baseline_score'].reset_index(drop=True).iloc[test_idx].to_numpy()
test_y = y.iloc[test_idx].to_numpy()

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))[:min(k, len(scores))]
    return float(np.asarray(labels)[order].mean())

metrics = []
for name, scores in [('Rule baseline', baseline_scores), ('Logistic model', model_scores)]:
    row = {'method': name, 'base_rate': float(test_y.mean()), 'roc_auc': float(roc_auc_score(test_y, scores)), 'average_precision': float(average_precision_score(test_y, scores))}
    for k in (10, 50, 100):
        row[f'precision_at_{k}'] = precision_at_k(test_y, scores, k)
    metrics.append(row)
metrics_df = pd.DataFrame(metrics)
display(metrics_df.round(3))
print(f'Train rows: {len(train_idx):,}; test rows: {len(test_idx):,}; held-out clients: {groups.iloc[test_idx].nunique()}')


## 4. Leakage sentinel

`trend_pct` is label-derived and must not be a feature. The sentinel below uses the negative of `trend_pct` because more negative trend values correspond to the positive decline class. A strong score here confirms that the test harness can detect the leak; this number must not be reported as model performance.

In [ ]:
leaky_score = -pd.to_numeric(lane['trend_pct'], errors='coerce').fillna(0).reset_index(drop=True).iloc[test_idx].to_numpy()
leak_auc = roc_auc_score(test_y, leaky_score)
print(f'Deliberate trend-derived leakage AUC: {leak_auc:.3f} (sentinel only)')
assert leak_auc > 0.80, 'Leakage sentinel failed; inspect the target or test harness.'
print('PASS: trend_pct is detectable as leakage and is excluded from the model.')


## 5. Limitations and recommendations

This is a retrospective starter-snapshot analysis, not a randomized content experiment. It cannot establish that refreshing a page causes more visibility, clicks, or engagement. The safe claim is observed, directional decision support for human review.

Recommended order: (1) review stale-and-visible pages first; (2) verify seasonality and recent editorial work manually; (3) monitor low-volume stale pages; (4) use reason codes as explanations, not automatic instructions; and (5) validate on a later time window before operational adoption.

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

In [ ]:
from pathlib import Path
out_dir = Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(out_dir / 'capstone_metrics.csv', index=False)
baseline_queue[['rank', 'baseline_score', 'baseline_action', 'baseline_reason', 'days_since_last_update', 'impressions_90d']].head(100).to_csv(out_dir / 'capstone_queue_preview.csv', index=False)
print(f'Wrote {out_dir / "capstone_metrics.csv"}')
print(f'Wrote {out_dir / "capstone_queue_preview.csv"}')


## Self-check

- [x] Data loads from the public raw GitHub URL.
- [x] Missing values are handled inside the model pipeline.
- [x] Grouped split is guaranteed to contain both classes.
- [x] Baseline and model use identical test rows.
- [x] Base rate and precision-at-K are reported.
- [x] Leakage sentinel correctly reverses trend_pct before testing.
- [ ] Run all cells and save the executed notebook before submission.